# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}\nPublished: {metadata.datePublished}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Get record sets and fields from Croissant schema
record_sets = list(dataset.record_sets)

print("Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# Preview fields and columns for each record set
for rs in record_sets:
    print(f"\nFields in RecordSet {rs['@id']}:")
    for field in rs.get('field', []):
        print(f"  - Field @id: {field['@id']}, name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')}")
    if 'column' in rs:
        print(f"Columns in RecordSet {rs['@id']}:")
        for col in rs['column']:
            print(f"  - Column @id: {col['@id']}, name: {col.get('name', 'N/A')}, dataType: {col.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nLoaded DataFrame for RecordSet {record_set_id}: Columns -> {df.columns.tolist()}")
    print(df.head())

# Choose the first record set for demonstration
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if main_record_set_id:
    print(f"\nDemonstration DataFrame columns for {main_record_set_id}: {dataframes[main_record_set_id].columns.tolist()}")
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA using GAD-7 total score and grouping by gender
# These field @ids are determined from previous overview, adjust as needed
gad7_field_id = 'https://api.app.sen.science/frontiers/7160411/fair2/gad7_total_score'  # Example @id
gender_field_id = 'https://api.app.sen.science/frontiers/7160411/fair2/gender'  # Example @id
record_set_id = main_record_set_id

df = dataframes[record_set_id]

# Ensure that GAD-7 score is numeric
if gad7_field_id in df.columns:
    df[gad7_field_id] = pd.to_numeric(df[gad7_field_id], errors='coerce')

    threshold = 10
    filtered_df = df[df[gad7_field_id] > threshold]
    print(f"Filtered records with {gad7_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize GAD-7 score
    filtered_df[f"{gad7_field_id}_normalized"] = (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) / filtered_df[gad7_field_id].std()
    print(f"Normalized {gad7_field_id} for filtered records:")
    print(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

    # Group by gender if available
    if gender_field_id in df.columns:
        grouped_df = filtered_df.groupby(gender_field_id)[gad7_field_id].mean().reset_index()
        print(f"Grouped data by {gender_field_id} (mean {gad7_field_id}):")
        print(grouped_df.head())
else:
    print(f"Field {gad7_field_id} not found in DataFrame columns for {record_set_id}. Adjust field @ids as needed.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization: distribution of GAD-7 total score
if main_record_set_id and gad7_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[gad7_field_id].dropna(), bins=10, kde=True)
    plt.title("Distribution of GAD-7 Total Score")
    plt.xlabel("GAD-7 Total Score")
    plt.ylabel("Count")
    plt.show()

# Boxplot of GAD-7 by gender
if gender_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=df, x=gender_field_id, y=gad7_field_id)
    plt.title("GAD-7 Total Score by Gender")
    plt.xlabel("Gender")
    plt.ylabel("GAD-7 Total Score")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded the dataset via its Croissant schema using mlcroissant.
- Reviewed available record sets, fields, and columns via their `@id`.
- Extracted and analyzed the main record set, focusing on GAD-7 total score and gender breakdown.
- Applied basic EDA, filtering records, normalizing scores, and visualizing distributions.
- This template provides a foundation for deeper AI-ready analyses and can be extended to other fields in the dataset by referencing their `@id`s.